# Demo 03: Multi-Agent Validation

Judgement questions ("best hotel for a honeymoon?") have no single database answer, so a model guesses, confidently. A **consortium** asks the same question to several different models and checks if they agree: agreement means confident, disagreement means they are guessing (likely hallucination). Details in the [README](./README.md).

In [ ]:
# !pip install -q -r requirements.txt   # self-paced only; pre-installed at AWS events
import os
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("AWS_REGION", "us-east-1")
from dotenv import load_dotenv
load_dotenv()

## Part 1: Build the consortium

Three models from two families (Claude and Nova) run the same question in parallel as a Strands **Graph**. We use a Graph, not a Swarm, so the models answer independently: a Swarm shares context between agents, so they would copy each other and lose the diversity the consortium needs. Neo4j (read-only) supplies the hotel list.

In [ ]:
import math
from collections import Counter
from strands import Agent
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder
from neo4j import GraphDatabase, RoutingControl

CONSORTIUM = {
    "claude-haiku": "global.anthropic.claude-haiku-4-5-20251001-v1:0",
    "nova-pro": "amazon.nova-pro-v1:0",
    "nova-lite": "us.amazon.nova-2-lite-v1:0",
}
ADVISOR = ("You are a travel agent that helps people with hotels. Answer by naming "
           "exactly one hotel from the list in the task and nothing else.")

_driver = GraphDatabase.driver(os.getenv("NEO4J_URI", "neo4j://127.0.0.1:7687"),
    auth=(os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "password")))
recs, _, _ = _driver.execute_query(
    "MATCH (h:Hotel) RETURN h.name AS name, h.city AS city, h.guestRating AS rating "
    "ORDER BY rating DESC LIMIT 12", routing_=RoutingControl.READ)
HOTELS = [r["name"] for r in recs]
CONTEXT = "\n".join(f"- {r['name']} ({r['city']}, rating {r['rating']})" for r in recs)

# Build the graph: one node per model, all run in parallel on the same task.
builder = GraphBuilder()
for name, model_id in CONSORTIUM.items():
    # callback_handler=None silences streaming so only our summary prints
    builder.add_node(Agent(name=name, system_prompt=ADVISOR, callback_handler=None,
        model=BedrockModel(model_id=model_id, region_name=os.environ["AWS_REGION"])), name)
    builder.set_entry_point(name)
graph = builder.build()

def verdict(question):
    result = graph(f"Hotels:\n{CONTEXT}\n\nQuestion: {question}")
    def canon(t):
        hits = [h for h in HOTELS if h.lower() in t.lower()]
        return max(hits, key=len) if hits else "(no answer)"
    answers = {name: canon(str(node.result)) for name, node in result.results.items()}
    tokens = {name: node.result.metrics.accumulated_usage for name, node in result.results.items()}
    votes = Counter(answers.values()); n = sum(votes.values())
    entropy = -sum((c/n) * math.log2(c/n) for c in votes.values()) / (math.log2(len(answers)) or 1)
    print("answers: ", answers)
    tok = {name: f"{u['inputTokens']}in/{u['outputTokens']}out" for name, u in tokens.items()}
    g = result.accumulated_usage
    print("tokens:  ", tok, f"| consortium total: {g['inputTokens']}in/{g['outputTokens']}out")
    print("clusters:", dict(votes), "| agreement:", round(votes.most_common(1)[0][1]/n, 2),
          "| entropy:", round(entropy, 2))
    print("=>", "CONFIDENT (consensus)" if len(votes) == 1
          else "LIKELY HALLUCINATION (models disagree, verify)")

print(f"Consortium ready: {list(CONSORTIUM)}")

## Part 2: Grounded question (models agree)

In [ ]:
# The ratings point to one clear answer, so the models converge.
verdict("Which hotel is best for a luxury traveler?")

## Part 3: Speculative question (models disagree)

In [ ]:
# Nothing in the data decides this, so each model picks a different hotel and they diverge.
verdict("Which single hotel best matches the color purple?")

## 🎯 Try It Yourself

In [ ]:
verdict("Which hotel is best for a solo business trip?")
_driver.close()

Interactive REPL: `python chat.py`. Next: [Demo 04: Neurosymbolic Guardrails](../04-neurosymbolic-demo/).